# 4.6 Awareness - OLS analysis

Treatment effect on awareness of the energy/resource costs of the group's activities (attitude_q_6). Post-only outcome (EL survey only, no BL wave) so following this specification:

$$ Y_i = \beta_0 + \beta_1 \text{Treated}_i + \beta_2 X_i + \epsilon_i $$

Same specification and controls as the no-enumerator-FE spec in 4_4 (coefficient plot), with WCB and RI checks on `treated`.

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import pyfixest as pf

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_regression_table import make_regression_table
from inference_func import wild_cluster_bootstrap_pval, randomization_inference_pval

In [2]:
# Load data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""]  # Only treat empty strings as NaN
)

In [3]:
# Keep only labgroups that have both pre and post observations (needed for log_baseline_energy_use)
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = labgroup_counts[labgroup_counts == 2].index
df = df[df["labgroupid"].isin(labgroups_to_keep)].copy()

## (1) Prepare dataset

In [4]:
# Create (log) baseline energy use control
df["baseline_energy_use"] = df.groupby("labgroupid")["annual_electricity_total_mwh"].transform(
    lambda x: x[df["survey"] == "BL"].iloc[0]
)
df["log_baseline_energy_use"] = np.log(df["baseline_energy_use"])

# Log transform group size - no_researchers is heavily right-skewed
df["log_no_researchers"] = np.log(df["no_researchers"])

In [5]:
# Keep only one row per labgroupid - awareness is EL-only
df = df[df["survey"] == "EL"].copy()

# Create mnf_indicator variable
df["mnf_indicator"] = df["faculty"].apply(lambda x: 1 if x == "Faculty of Science (MNF)" else 0).astype(int)

In [6]:
# Recode awareness response to numeric scale, restrict to labs that answered this question
attitude_map = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Neither agree nor disagree": 3,
    "Agree": 4,
    "Strongly agree": 5,
}

df = df.dropna(subset=["attitude_q_6"]).copy()
df["y"] = df["attitude_q_6"].map(attitude_map)

print(f"{len(df)} lab groups answered the awareness question")

93 lab groups answered the awareness question


## (2) OLS regression

In [7]:
fit_awareness = pf.feols(
    "y ~ treated + log_no_researchers + log_baseline_energy_use + mnf_indicator",
    data=df,
    vcov={"CRV1": "enum_id"}
)
fit_awareness.summary()

###

Estimation:  OLS
Dep. var.: y
sample: None = all
Inference:  CRV1
Observations:  93

| Coefficient             |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept               |      2.902 |        0.384 |     7.550 |      0.000 |  2.115 |   3.689 |
| treated                 |      0.388 |        0.165 |     2.343 |      0.026 |  0.049 |   0.726 |
| log_no_researchers      |     -0.000 |        0.186 |    -0.002 |      0.998 | -0.382 |   0.381 |
| log_baseline_energy_use |      0.005 |        0.068 |     0.074 |      0.941 | -0.133 |   0.143 |
| mnf_indicator           |      0.583 |        0.216 |     2.702 |      0.012 |  0.141 |   1.024 |
---
RMSE: 1.008 R2: 0.102 


## (3) Wild cluster bootstrap and randomization inference

In [8]:
REPS = 999

wcb_pval = wild_cluster_bootstrap_pval(fit_awareness, param="treated", cluster="enum_id", reps=REPS, seed=config.SEED)

fit_fn = lambda d: pf.feols(
    "y ~ treated + log_no_researchers + log_baseline_energy_use + mnf_indicator",
    data=d, vcov={"CRV1": "enum_id"}
)
ri_pval, _ = randomization_inference_pval(
    df, fit_fn, param="treated", observed_coef=fit_awareness.coef().loc["treated"],
    reps=REPS, seed=config.SEED,
)

print(f"WCB p-value: {wcb_pval:.3f}")
print(f"RI p-value:  {ri_pval:.3f}")

WCB p-value: 0.026
RI p-value:  0.113


## (4) Make regression table

In [9]:
table = make_regression_table(
    fit_list      = [fit_awareness],
    model_names   = ["(1)"],
    keep_vars     = ["treated"],
    var_labels    = {"treated": "Treated"},
    fe_rows       = None,
    baseline_mean = None,
    decimals      = 3,
    r2_type       = None,
    wcb_pvals     = [wcb_pval],
    ri_pvals      = [ri_pval],
    col1_width    = "5.5cm",
    coln_width    = "3cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "awareness_ols_main.tex"
_ = table_path.write_text(table)
print(table)

\begin{tabular}{@{}L{5.5cm}C{3cm}}
\hline
\addlinespace[0.2cm]
 & (1) \\
\hline
\addlinespace[0.2cm]
Treated & $ 0.388\rlap{**}$ \\
 & ($ 0.165$) \\
\addlinespace[0.2cm]
\hline
\addlinespace[0.2cm]
WCB $p$-value & $ 0.026$ \\
RI $p$-value & $ 0.113$ \\
\addlinespace[0.2cm]
Number of observations & $ 93$ \\
\addlinespace[0.2cm]
\hline
\end{tabular}
